# TBD Phase 2 26L: Performance & Computing Models

## Introduction
In this lab, you will compare the performance and computing models of four popular data processing libraries/engines: **Polars, Pandas, DuckDB, and PySpark**.

You will explore:
- **Performance**: single-node processing speed, parallel execution, memory usage, and result materialization cost.
- **Scalability**: how performance changes with the number of local threads/cores and with Spark executors on a cluster.
- **Physical layout**: how file format, Parquet layout, row groups, sorting, partitioning, and pruning affect IO.
- **Computing models**: in-memory vs. out-of-core processing, SQL vs. DataFrame APIs, eager vs. lazy execution, and streaming execution vs. streaming output.

This notebook is an assignment template. It gives you a common structure and helper code, but you must design your own dataset variant, queries, benchmark implementation, and analysis.


## Submission identity

Before starting the assignment, copy this notebook into your fork of the workshop repository and work on that copy.

Fill in the first code cell with:

- your group number,
- a link to this notebook in your forked GitHub repository,
- names or IDs of group members if required by the instructor.

The submitted notebook should be reachable from your fork. Do not submit a notebook that only exists locally.

In [4]:
# TODO: Fill this in before submitting.
GROUP_ID = 1
# Prawdziwa to 17, ale nie ma dla niej wariantu
NOTEBOOK_URL = "https://github.com/soswi/MENG-PW-S1-TBD-WORKSHOP-1/blob/master/notebooks/tbd_phase_2_26L.ipynb"
GROUP_MEMBERS = [
    "Wiktor Sosnowski / 348561",
]

assert GROUP_ID is not None, "Set GROUP_ID before running the notebook"
assert "<your-github-user-or-org>" not in NOTEBOOK_URL, "Set NOTEBOOK_URL to your forked repository notebook URL"

## Library/engine capabilities

Use this table as a reference when interpreting your results.

| Library/engine | Query optimizer | Distributed | Arrow-backed | Out-of-core | Parallel local execution | Main APIs |
|---|---|---|---|---|---|---|
| **Pandas 3.0** | no | no | default IO returns NumPy-backed data; `dtype_backend="pyarrow"` returns PyArrow-backed nullable dtypes | no | limited | DataFrame, `pd.col` for selected expression-style usage |
| **Polars** | yes | single-node locally; distributed engine is available in Polars Cloud and is outside this local benchmark | yes | yes | yes | DataFrame, lazy expressions, SQL subset |
| **DuckDB** | yes | no | yes | yes | yes | SQL, relational API |
| **PySpark** | yes | yes | yes, for selected IO/UDF paths | yes | yes | SQL, DataFrame |

The goal is not to prove that one library is always best. The goal is to identify which library/engine is appropriate for a given data size, query shape, memory limit, physical layout, and deployment model.

Use pandas 3.0 in this lab. Two pandas 3.0 behaviours matter for the benchmark: string columns are no longer inferred as generic `object` dtype by default, and Copy-on-Write is the only mutation model. In addition, compare two Pandas Parquet-reading variants where possible:

- default Pandas/NumPy-backed DataFrame: `pd.read_parquet(path)`,
- PyArrow-backed DataFrame: `pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")`.

Record the pandas version and dtypes in your report.


## Prerequisites

Install the required libraries in your notebook environment. If the course image already contains them, this command should be quick. Pandas 3.0 requires Python 3.11 or newer.

Use current Polars API in new code. In particular, use `collect(engine="streaming")` for streaming execution and use sink methods when you want to write streaming output to disk.

For Pandas, benchmark both the default backend and the PyArrow dtype backend for Parquet reads. The PyArrow-backed variant is especially relevant for string-heavy datasets.


In [ ]:
%pip install -U "pandas>=3.0,<3.1" polars duckdb pyspark faker deltalake memory_profiler pyarrow psutil matplotlib seaborn

In [5]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

In [6]:
import gc
import os
import time
import json
import platform
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import polars as pl
import duckdb
import psutil
from faker import Faker
from memory_profiler import memory_usage
from pyspark.sql import SparkSession

print("Python:", platform.python_version())
if tuple(map(int, platform.python_version_tuple()[:2])) < (3, 11):
    raise RuntimeError("This notebook requires Python 3.11+ because it uses pandas 3.0.")
print("Polars:", pl.__version__)
print("Pandas:", pd.__version__)
if tuple(map(int, pd.__version__.split(".")[:2])) < (3, 0):
    raise RuntimeError("Install pandas 3.0+ before running the benchmark.")
print("DuckDB:", duckdb.__version__)
print("CPU logical cores:", psutil.cpu_count(logical=True))
print("RAM GiB:", round(psutil.virtual_memory().total / 2**30, 2))


Python: 3.12.3
Polars: 1.42.0
Pandas: 3.0.3
DuckDB: 1.5.4
CPU logical cores: 12
RAM GiB: 30.66


## Part 1: Data generation with group variants

Each group works with one assigned synthetic data profile. Use your group number to select the variant card below.

Your dataset does not need to match other groups exactly, but it must satisfy the common schema and benchmarking requirements described in this notebook.

Every group must document:
- dataset profile,
- main benchmark row count, plus any additional stress-test row counts if used,
- physical layout and file format choices,
- library versions,
- query intent,
- benchmark results,
- conclusions.

You may use the helper functions below, but you must adapt the dataset to your assigned variant.


## Variant cards for 16 groups

Choose or assign one variant per group.

| Group | Data profile | Required data feature | Suggested query stress |
|---:|---|---|---|
| 1 | Social media posts | tags or hashtags | explode/list handling, top-k |
| 2 | E-commerce orders | products and order values | join, category aggregation |
| 3 | IoT telemetry | device time series | time filters, rolling/window logic |
| 4 | Application logs | status codes and endpoints | selective filters, string columns |
| 5 | Advertising clicks | campaign skew | CTR, skewed group-by, join |
| 6 | Game events | player sessions | high-cardinality group-by |
| 7 | Streaming platform events | watch duration | device/country aggregation |
| 8 | Public transport events | route delays | time and location aggregation |
| 9 | Banking-like transactions | risk/fraud flags | selective filters, top-k, sorting |
| 10 | Web analytics | referrers and pages | funnel-like aggregation |
| 11 | Delivery/logistics events | late status updates | late events, time windows |
| 12 | Education platform activity | courses and students | joins and progress metrics |
| 13 | Weather measurements | missing values | resampling and null handling |
| 14 | Marketplace listings | prices and categories | quantiles, category statistics |
| 15 | Security events | rare alerts | selective filters and high skew |
| 16 | Support tickets | priority and SLA | time-to-resolution metrics |

You may rename columns and categories to fit the chosen profile. Keep enough common structure to run the same engine comparisons.

In [7]:
DOMAIN_CARDS = {
    1: {"name": "Social media posts", "feature": "tags", "stress": "explode/list handling and top-k"},
    2: {"name": "E-commerce orders", "feature": "products", "stress": "joins and category aggregation"},
    3: {"name": "IoT telemetry", "feature": "device time series", "stress": "time filters and rolling/window logic"},
    4: {"name": "Application logs", "feature": "status codes", "stress": "selective filters and string columns"},
    5: {"name": "Advertising clicks", "feature": "campaign skew", "stress": "CTR, skewed group-by, and joins"},
    6: {"name": "Game events", "feature": "player sessions", "stress": "high-cardinality group-by"},
    7: {"name": "Streaming platform events", "feature": "watch duration", "stress": "device/country aggregation"},
    8: {"name": "Public transport events", "feature": "route delays", "stress": "time and location aggregation"},
    9: {"name": "Banking-like transactions", "feature": "risk flags", "stress": "selective filters, top-k, and sorting"},
    10: {"name": "Web analytics", "feature": "referrers", "stress": "funnel-like aggregation"},
    11: {"name": "Delivery/logistics events", "feature": "late status updates", "stress": "late events and time windows"},
    12: {"name": "Education platform activity", "feature": "courses", "stress": "joins and progress metrics"},
    13: {"name": "Weather measurements", "feature": "missing values", "stress": "resampling and null handling"},
    14: {"name": "Marketplace listings", "feature": "prices", "stress": "quantiles and category statistics"},
    15: {"name": "Security events", "feature": "rare alerts", "stress": "selective filters and high skew"},
    16: {"name": "Support tickets", "feature": "priority and SLA", "stress": "time-to-resolution metrics"},
}

assert 1 <= GROUP_ID <= 16, "GROUP_ID must be between 1 and 16"
CARD = DOMAIN_CARDS[GROUP_ID]
CARD

{'name': 'Social media posts',
 'feature': 'tags',
 'stress': 'explode/list handling and top-k'}

## Dataset requirements

Your generated dataset must contain at least:

- one timestamp column,
- one high-cardinality identifier, such as user, device, session, order, ticket, or transaction id,
- at least two categorical columns,
- at least two numeric metric columns,
- one feature specific to your variant card,
- enough rows to make local benchmark differences visible,
- a Parquet output file or directory.

Recommended starting sizes:

| Scale | Rows | Use case |
|---|---:|---|
| debug | 200,000 | Validate code quickly |
| small | 2,000,000 | Local development and first benchmark |
| medium | 10,000,000 to 20,000,000 | Main benchmark |
| large | 50,000,000+ | Optional stress test |

Use `debug` only while developing. The rendered notebook should report one main benchmark size (`N_ROWS`). If you run additional sizes, put those results in a separate stress-test table and do not mix them with the main benchmark table.

It is acceptable for different groups to generate different random data. Choose one main dataset size for the benchmark and record it as `N_ROWS`. You may use smaller debug data while developing and optional larger data for stress tests, but those extra sizes should be reported separately.

In [8]:
# TODO: Choose the main dataset scale for your final benchmark and verify output paths before generation.
# N_ROWS is the main row count reported for this notebook. Extra row counts are optional stress tests.
# Dataset configuration
SCALE = "small"
SCALE_ROWS = {
    "debug": 200_000,
    "small": 2_000_000,
    "medium": 10_000_000,
    "large": 50_000_000,
}

N_ROWS = SCALE_ROWS[SCALE]
OUTPUT_DIR = Path("../data/phase2_26L") / f"group_{GROUP_ID:02d}"
EVENTS_PATH = OUTPUT_DIR / "events.parquet"
PARTITIONED_EVENTS_DIR = OUTPUT_DIR / "events_partitioned"
OPTIMIZED_EVENTS_PATH = OUTPUT_DIR / "events_optimized.parquet"
DIMENSION_PATH = OUTPUT_DIR / "dimension.parquet"
MANIFEST_PATH = OUTPUT_DIR / "manifest.json"

# Required negative baseline paths for the file-format/layout task. Do not commit these generated files.
CSV_EVENTS_PATH = OUTPUT_DIR / "events.csv"
JSON_EVENTS_PATH = OUTPUT_DIR / "events.jsonl"

# Leave SEED as None if you want independent data on each generation.
# If you need to reproduce exactly the same dataset later, set SEED to the value stored in the manifest.
SEED = None
RUN_SEED = int(np.random.SeedSequence().entropy) if SEED is None else int(SEED)
rng = np.random.default_rng(RUN_SEED)
fake = Faker()

print("Group:", GROUP_ID, CARD)
print("Rows:", N_ROWS)
print("Run seed recorded in manifest:", RUN_SEED)
print("Output directory:", OUTPUT_DIR)


Group: 1 {'name': 'Social media posts', 'feature': 'tags', 'stress': 'explode/list handling and top-k'}
Rows: 2000000
Run seed recorded in manifest: 124202522203822218539711678633365616362
Output directory: ../data/phase2_26L/group_01


## Generator template

The helper below creates a common base event table. You should extend it for your variant.

Do not spend most of the assignment writing a perfect data generator. The generator only needs to create data that is large enough and structurally interesting enough for your benchmark questions.

In [9]:
# TODO: Adapt customize_for_variant(...) and generate_dimension_table(...) to your variant.
def skewed_ids(rng, n, max_id, hot_fraction=0.02, hot_probability=0.50):
    hot_count = max(1, int(max_id * hot_fraction))
    ids = rng.integers(hot_count + 1, max_id + 1, size=n)
    hot_mask = rng.random(n) < hot_probability
    ids[hot_mask] = rng.integers(1, hot_count + 1, size=hot_mask.sum())
    return ids


def random_tag_lists(rng, n, vocabulary=None, min_tags=1, max_tags=3):
    vocabulary = np.array(vocabulary or ["ai", "cloud", "spark", "polars", "duckdb", "sql", "etl", "security", "mlops"])
    counts = rng.integers(min_tags, max_tags + 1, size=n)
    tag_ids = rng.integers(0, len(vocabulary), size=(n, max_tags))
    return [[str(vocabulary[tag_ids[i, j]]) for j in range(counts[i])] for i in range(n)]


def generate_base_events(n, rng):
    start = np.datetime64("2026-01-01T00:00:00", "s")
    end = np.datetime64("2026-04-01T00:00:00", "s")
    seconds = int((end - start) / np.timedelta64(1, "s"))
    event_ts = (start + rng.integers(0, seconds, size=n).astype("timedelta64[s]")).astype("datetime64[us]")

    df = pl.DataFrame(
        {
            "event_id": np.arange(1, n + 1),
            "entity_id": skewed_ids(rng, n, max_id=200_000),
            "event_ts": event_ts,
            "category": rng.choice(["A", "B", "C", "D", "E", "F"], size=n),
            "country": rng.choice(["PL", "DE", "FR", "UK", "US", "IN", "BR"], size=n),
            "device": rng.choice(["mobile", "desktop", "tablet"], size=n, p=[0.65, 0.25, 0.10]),
            "metric_1": rng.lognormal(mean=4.0, sigma=1.0, size=n).round(3),
            "metric_2": rng.integers(0, 10_000, size=n),
            "tags": random_tag_lists(rng, n),
        }
    )
    return df.with_columns(pl.col("event_ts").dt.date().alias("event_date"))


# Hashtag vocabulary for group 1, Social media posts variant.
SOCIAL_TAGS = [
    "travel", "food", "music", "sports", "tech", "fashion", "gaming",
    "fitness", "art", "news", "comedy", "nature", "politics", "movies", "books",
]

# Post types for the social media domain. Story and reel are rarer than post and video.
POST_TYPES = ["post", "photo", "video", "story", "reel"]
POST_TYPE_PROBS = [0.40, 0.25, 0.20, 0.10, 0.05]


def customize_for_variant(df, card, rng):
    # Adapted for variant 1 (Social media posts). Changes made:
    # - renamed entity_id to user_id, category to post_type, metric_1/metric_2 to engagement_score/char_count,
    # - added domain-specific post_type categories (post/photo/video/story/reel),
    # - added two numeric columns: likes and shares,
    # - introduced nulls in char_count,
    # - tags column reuses the variant's hashtag vocabulary (required feature for this variant).
    n = df.height

    df = df.rename({"entity_id": "user_id", "category": "post_type", "metric_1": "engagement_score", "metric_2": "char_count"})

    df = df.with_columns(pl.Series("post_type", rng.choice(POST_TYPES, size=n, p=POST_TYPE_PROBS)))

    df = df.with_columns(pl.Series("tags", random_tag_lists(rng, n, vocabulary=SOCIAL_TAGS, min_tags=1, max_tags=4)))

    # Likes and shares: lognormal, most posts get few interactions, a small number go viral.
    likes = rng.lognormal(mean=2.0, sigma=1.5, size=n).round().astype("int64")
    shares = (likes * rng.uniform(0.0, 0.3, size=n)).round().astype("int64")
    df = df.with_columns(pl.Series("likes", likes), pl.Series("shares", shares))

    # 1% of posts have no recorded char_count, for null handling exercises.
    null_mask = pl.Series(rng.random(n) < 0.01)
    df = df.with_columns(
        pl.when(null_mask).then(None).otherwise(pl.col("char_count")).alias("char_count")
    )

    return df


def generate_dimension_table(card, rng):
    # Dimension table of creator accounts, for the join query (user_id -> account tier, follower_count).
    n_accounts = 1_000
    return pl.DataFrame(
        {
            "user_id": np.arange(1, n_accounts + 1),
            "account_tier": rng.choice(["free", "creator", "business"], size=n_accounts, p=[0.7, 0.2, 0.1]),
            "follower_count": rng.lognormal(mean=6.0, sigma=2.0, size=n_accounts).round().astype("int64"),
        }
    )

In [10]:
# TODO: Run this after adapting the generator. Verify that generated data is not committed to Git.
# Generate and save the dataset
print("Start Generate and save the dataset")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

base_events = generate_base_events(N_ROWS, rng)
events = customize_for_variant(base_events, CARD, rng)
dimension = generate_dimension_table(CARD, rng)

events.write_parquet(EVENTS_PATH, compression="zstd")
dimension.write_parquet(DIMENSION_PATH, compression="zstd")

# Optional partitioned layout for experiments with predicate pushdown and file layout.
events.write_parquet(PARTITIONED_EVENTS_DIR, partition_by="event_date", compression="zstd")

# TODO: Create an optimized Parquet layout for one selected query pattern.
# Example ideas:
# - sort by columns used in range filters before writing,
# - choose a smaller row_group_size if it improves row-group pruning,
# - partition by date or another selective filter column,
# - add bloom filters only if your chosen writer and reader expose this option clearly.
# Replace the sort columns with columns from your own query pattern.
# events.sort(["event_date", "category"]).write_parquet(
#     OPTIMIZED_EVENTS_PATH,
#     compression="zstd",
#     row_group_size=100_000,
# )

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "group_id": GROUP_ID,
    "variant": CARD,
    "scale": SCALE,
    "rows": int(events.height),
    "run_seed": RUN_SEED,
    "paths": {
        "events": str(EVENTS_PATH),
        "events_partitioned": str(PARTITIONED_EVENTS_DIR),
        "events_optimized": str(OPTIMIZED_EVENTS_PATH),
        "dimension": str(DIMENSION_PATH),
    },
    "environment": {
        "python": platform.python_version(),
        "polars": pl.__version__,
        "pandas": pd.__version__,
        "duckdb": duckdb.__version__,
        "cpu_logical_cores": psutil.cpu_count(logical=True),
        "ram_gib": round(psutil.virtual_memory().total / 2**30, 2),
    },
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(json.dumps(manifest, indent=2))


Start Generate and save the dataset
{
  "created_at_utc": "2026-06-28T15:46:41.579181+00:00",
  "group_id": 1,
  "variant": {
    "name": "Social media posts",
    "feature": "tags",
    "stress": "explode/list handling and top-k"
  },
  "scale": "small",
  "rows": 2000000,
  "run_seed": 124202522203822218539711678633365616362,
  "paths": {
    "events": "../data/phase2_26L/group_01/events.parquet",
    "events_partitioned": "../data/phase2_26L/group_01/events_partitioned",
    "events_optimized": "../data/phase2_26L/group_01/events_optimized.parquet",
    "dimension": "../data/phase2_26L/group_01/dimension.parquet"
  },
  "environment": {
    "python": "3.12.3",
    "polars": "1.42.0",
    "pandas": "3.0.3",
    "duckdb": "1.5.4",
    "cpu_logical_cores": 12,
    "ram_gib": 30.66
  }
}


## Dataset sanity checks

Before benchmarking, inspect your schema and basic statistics. Your report should briefly explain why your dataset is suitable for the queries you chose.

In [11]:
# TODO: Inspect schema, row count, null counts, and basic category distributions.
# Keep this section short, but include enough evidence that your data was generated correctly.
print(events.schema)
print(events.head(5))
print(events.null_count())

Schema({'event_id': Int64, 'user_id': Int64, 'event_ts': Datetime(time_unit='us', time_zone=None), 'post_type': String, 'country': String, 'device': String, 'engagement_score': Float64, 'char_count': Int64, 'tags': List(String), 'event_date': Date, 'likes': Int64, 'shares': Int64})
shape: (5, 12)
┌──────────┬─────────┬───────────────┬───────────┬───┬────────────┬────────────┬───────┬────────┐
│ event_id ┆ user_id ┆ event_ts      ┆ post_type ┆ … ┆ tags       ┆ event_date ┆ likes ┆ shares │
│ ---      ┆ ---     ┆ ---           ┆ ---       ┆   ┆ ---        ┆ ---        ┆ ---   ┆ ---    │
│ i64      ┆ i64     ┆ datetime[μs]  ┆ str       ┆   ┆ list[str]  ┆ date       ┆ i64   ┆ i64    │
╞══════════╪═════════╪═══════════════╪═══════════╪═══╪════════════╪════════════╪═══════╪════════╡
│ 1        ┆ 1291    ┆ 2026-03-29    ┆ photo     ┆ … ┆ ["music",  ┆ 2026-03-29 ┆ 6     ┆ 1      │
│          ┆         ┆ 02:30:49      ┆           ┆   ┆ "art", …   ┆            ┆       ┆        │
│          ┆    

In [12]:
print((events["char_count"].is_nan()).sum())
print(events["char_count"].dtype)
print(events.filter(pl.col("char_count").is_null()).height)
print(events.select(pl.col("char_count").null_count()))

0
Int64
20117
shape: (1, 1)
┌────────────┐
│ char_count │
│ ---        │
│ u32        │
╞════════════╡
│ 20117      │
└────────────┘


In [13]:
print("Shape:", events.shape)
print()
print("Schema:")
print(events.schema)
print()
print("Null counts:")
for col in events.columns:
    print(f"  {col}: {events.select(pl.col(col).null_count()).item()}")
print()
print("post_type distribution:")
print(events["post_type"].value_counts().sort("post_type"))
print()
print("country distribution:")
print(events["country"].value_counts().sort("country"))
print()
print("user_id cardinality:", events["user_id"].n_unique())
print("event_date range:", events["event_date"].min(), "to", events["event_date"].max())

Shape: (2000000, 12)

Schema:
Schema({'event_id': Int64, 'user_id': Int64, 'event_ts': Datetime(time_unit='us', time_zone=None), 'post_type': String, 'country': String, 'device': String, 'engagement_score': Float64, 'char_count': Int64, 'tags': List(String), 'event_date': Date, 'likes': Int64, 'shares': Int64})

Null counts:
  event_id: 0
  user_id: 0
  event_ts: 0
  post_type: 0
  country: 0
  device: 0
  engagement_score: 0
  char_count: 20117
  tags: 0
  event_date: 0
  likes: 0
  shares: 0

post_type distribution:
shape: (5, 2)
┌───────────┬────────┐
│ post_type ┆ count  │
│ ---       ┆ ---    │
│ str       ┆ u32    │
╞═══════════╪════════╡
│ photo     ┆ 500208 │
│ post      ┆ 799595 │
│ reel      ┆ 100218 │
│ story     ┆ 199763 │
│ video     ┆ 400216 │
└───────────┴────────┘

country distribution:
shape: (7, 2)
┌─────────┬────────┐
│ country ┆ count  │
│ ---     ┆ ---    │
│ str     ┆ u32    │
╞═════════╪════════╡
│ BR      ┆ 285699 │
│ DE      ┆ 285404 │
│ FR      ┆ 285095 │
│ IN

## Part 2: Measuring performance

You must use one consistent benchmark protocol for all libraries/engines.

Minimum requirements:

1. Run every benchmark at least three times. Five repetitions are recommended.
2. Run `gc.collect()` before each measured repetition to reduce noise from previous Python allocations.
3. Report median runtime, not only one measurement.
4. Record peak memory where possible.
5. Check that results are logically equivalent across libraries/engines.
6. Store your results in a table.
7. Describe any library/engine-specific settings, such as Pandas dtype backend, thread count, Spark local mode, or DuckDB threads.

**Important for memory benchmarks**: notebook kernels keep allocations and library state between cells. Peak-RSS comparisons are often misleading when all variants run in the same process. For Task 3.1 and any memory-sensitive comparison, prefer running each variant in a fresh process or a small standalone script. If you cannot do that, clearly state this limitation.

You may use the helper shape below, but you need to implement the actual benchmark functions.


In [14]:
# TODO: Implement or adapt benchmark helpers before collecting final results.
BENCHMARK_COLUMNS = [
    "library_engine",
    "mode",
    "query_name",
    "data_format",
    "layout",
    "rows",
    "median_time_s",
    "peak_memory_mb",
    "input_size_mb",
    "result_check",
    "notes",
]

benchmark_results = []

# TODO: Implement your timing and memory measurement helper.
# Suggested output: one dictionary matching BENCHMARK_COLUMNS per library/engine/query/mode.
# Recommended inside each measured repetition:
# gc.collect()
# start = time.perf_counter()
def run_benchmark(library_engine, mode, query_name, data_format, layout, fn, repeats=5, notes=""):
    """
    Run fn() `repeats` times, measure wall time and peak RSS via memory_profiler.
    fn must return (result_object, rows, input_size_mb, result_check_value).
    Appends median results to benchmark_results and returns the last result_object.

    Limitation: this runs all repeats in the same notebook kernel/process, so
    peak memory reflects cumulative process RSS, not an isolated measurement per
    library. Library caches and prior allocations from earlier cells can inflate
    or hide the real peak. For Task 3.1 and other memory-sensitive comparisons,
    run each mode in a fresh process instead.
    """
    times = []
    peak_memories = []
    result_obj = None
    rows = None
    input_size_mb = None
    result_check = None

    def _call():
        nonlocal result_obj, rows, input_size_mb, result_check
        result_obj, rows, input_size_mb, result_check = fn()

    for i in range(repeats):
        gc.collect()
        start = time.perf_counter()
        mem_samples = memory_usage((_call, (), {}), interval=0.05, max_usage=True, retval=False)
        elapsed = time.perf_counter() - start

        times.append(elapsed)
        peak_memories.append(mem_samples if isinstance(mem_samples, float) else max(mem_samples))

    median_time = float(np.median(times))
    median_peak_mb = float(np.median(peak_memories))

    benchmark_results.append({
        "library_engine": library_engine,
        "mode": mode,
        "query_name": query_name,
        "data_format": data_format,
        "layout": layout,
        "rows": rows,
        "median_time_s": round(median_time, 4),
        "peak_memory_mb": round(median_peak_mb, 2),
        "input_size_mb": round(input_size_mb, 2) if input_size_mb is not None else None,
        "result_check": result_check,
        "notes": notes,
    })

    print(f"{library_engine} / {mode} / {query_name}: median {median_time:.4f}s, peak_RSS {median_peak_mb:.2f}MB (n={repeats})")
    return result_obj

## Part 3: Student tasks

### Task 1: Design three benchmark queries

Create three queries of your own choice. They must test different behavior.

Your queries should cover at least three of the following classes:

- selective filter plus aggregation,
- high-cardinality group-by,
- top-k or sorting,
- list/tag explode,
- join with a dimension table,
- window or rolling computation,
- query that produces a large output,
- query sensitive to partitioned vs. unpartitioned layout,
- query sensitive to column pruning, predicate pushdown, or row-group pruning.

For each query, write a short hypothesis before you run it:

- what does this query test?
- which library/engine do you expect to perform best?
- which library/engine may use the most memory?
- which physical layout should help, if any?


In [47]:
# TODO: Define your three query specifications in prose or structured metadata.
# Do not start benchmarking before you can explain what each query is supposed to test.

Query 1: tag explode + top-k

Tag explode + top-k. Explode the tags list column (one row per tag), count tag frequency, return top 20 tags.

What does this query test: native list/array handling and explode performance.
Expected best engine: DuckDB or Polars (columnar explode is cheap for both).
Expected highest memory: Pandas, since list-typed columns are stored as Python object arrays, and exploding them multiplies row count without columnar storage.
Layout that should help: none in particular, the whole tags column must be read regardless of partitioning.
Query 2: selective filter + aggregation

Selective filter + aggregation by post_type and country. Filter posts with likes > 100 (viral, a small selective subset) from March 2026, group by post_type and country, compute sum(likes), avg(engagement_score), count.

What does this query test: predicate pushdown and column pruning on a selective filter.
Expected best engine: DuckDB and Polars lazy (query optimizer can push the filter into the Parquet scan). PySpark should also pushdown the predicate well.
Expected highest memory: Pandas default backend, since it materializes the full scan result in memory before filtering.
Layout that should help: partitioning by event_date (file pruning) or a sorted layout with a smaller row_group_size (row-group pruning).
Query 3: join + top-k by tier

Join with dimension table + top-k by follower tier. Join events with the dimension table on user_id, group by account_tier, compute sum(likes) and sum(shares), then return the top 10 user_id by likes within account_tier == 'business'.

What does this query test: join cost, high-cardinality group-by, and sorting a large intermediate result.
Expected best engine at this dataset size (2M rows): DuckDB or Polars, since the join fits comfortably in memory with no JVM/scheduler startup cost.
Expected highest memory: PySpark (JVM heap and serialization overhead) or Pandas default backend (materializes the full join before aggregating).
Layout that should help: none specific; this query is driven by join and group-by cost rather than IO layout.

### Task 2: Benchmark local libraries/engines

Implement your three queries in:

- Pandas 3.0 with the default NumPy-backed output from `pd.read_parquet(path)`,
- Pandas 3.0 with `pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")`,
- Polars,
- DuckDB,
- PySpark local mode.

For Polars, benchmark at least:

- eager execution,
- lazy execution with default collection,
- lazy execution with streaming engine.

For PySpark, use local mode in this task. Dataproc is a separate task later in the notebook.


In [17]:
# TODO: Configure Spark local only when you start the PySpark local benchmark.
# Initialize Spark only when you start the Spark part of the benchmark.
# TODO: Adjust memory and local core count if needed.

spark = (
    SparkSession.builder
    .appName("TBDPhase2LocalBenchmark")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/28 15:51:00 WARN Utils: Your hostname, serverubuntuprodesk405g6, resolves to a loopback address: 127.0.1.1; using 192.168.0.200 instead (on interface eno1)
26/06/28 15:51:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/28 15:51:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [18]:
# TODO: Pandas implementations of your three queries.
# Implement both Pandas read variants:
# 1. default backend: pd.read_parquet(path)
# 2. PyArrow backend: pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")
#
# Report dtypes for both variants and compare runtime/memory.

# Pandas default backend (NumPy-backed)
pdf_default = pd.read_parquet(EVENTS_PATH)
print("Pandas default dtypes:")
print(pdf_default.dtypes)
print()

# Pandas PyArrow backend
pdf_arrow = pd.read_parquet(EVENTS_PATH, engine="pyarrow", dtype_backend="pyarrow")
print("Pandas PyArrow dtypes:")
print(pdf_arrow.dtypes)

Pandas default dtypes:
event_id                     int64
user_id                      int64
event_ts            datetime64[us]
post_type                      str
country                        str
device                         str
engagement_score           float64
char_count                 float64
tags                        object
event_date                  object
likes                        int64
shares                       int64
dtype: object

Pandas PyArrow dtypes:
event_id                                        int64[pyarrow]
user_id                                         int64[pyarrow]
event_ts                                timestamp[us][pyarrow]
post_type                                large_string[pyarrow]
country                                  large_string[pyarrow]
device                                   large_string[pyarrow]
engagement_score                               double[pyarrow]
char_count                                      int64[pyarrow]
tags           

In [19]:
def pandas_query1(df, backend_name):
    exploded = df.explode("tags")
    result = exploded["tags"].value_counts().head(20)
    return result, len(df), os.path.getsize(EVENTS_PATH) / (1024 * 1024), len(result)


run_benchmark(
    library_engine="pandas_default",
    mode="default",
    query_name="q1_tag_explode_topk",
    data_format="parquet",
    layout="default",
    fn=lambda: pandas_query1(pdf_default, "default"),
)

run_benchmark(
    library_engine="pandas_pyarrow",
    mode="pyarrow_backend",
    query_name="q1_tag_explode_topk",
    data_format="parquet",
    layout="default",
    fn=lambda: pandas_query1(pdf_arrow, "pyarrow"),
)

pandas_default / default / q1_tag_explode_topk: median 2.3824s, peak_RSS 2365.18MB (n=5)
pandas_pyarrow / pyarrow_backend / q1_tag_explode_topk: median 0.7865s, peak_RSS 2283.27MB (n=5)


tags
tech        334175
news        334120
politics    333963
food        333839
movies      333714
nature      333662
fashion     333590
comedy      333549
gaming      333249
books       333141
fitness     333100
sports      332872
music       332834
art         332753
travel      332279
Name: count, dtype: int64[pyarrow]

In [20]:
def pandas_query2(df, backend_name):
    filtered = df[(df["likes"] > 100) & (df["event_date"] >= pd.Timestamp("2026-03-01").date()) & (df["event_date"] <= pd.Timestamp("2026-03-31").date())]
    result = filtered.groupby(["post_type", "country"]).agg(
        sum_likes=("likes", "sum"),
        avg_engagement=("engagement_score", "mean"),
        cnt=("likes", "count"),
    ).reset_index()
    return result, len(df), os.path.getsize(EVENTS_PATH) / (1024 * 1024), len(result)


run_benchmark(
    library_engine="pandas_default",
    mode="default",
    query_name="q2_filter_aggregation",
    data_format="parquet",
    layout="default",
    fn=lambda: pandas_query2(pdf_default, "default"),
)

run_benchmark(
    library_engine="pandas_pyarrow",
    mode="pyarrow_backend",
    query_name="q2_filter_aggregation",
    data_format="parquet",
    layout="default",
    fn=lambda: pandas_query2(pdf_arrow, "pyarrow"),
)

pandas_default / default / q2_filter_aggregation: median 0.3724s, peak_RSS 2179.32MB (n=5)
pandas_pyarrow / pyarrow_backend / q2_filter_aggregation: median 0.1963s, peak_RSS 2181.27MB (n=5)


,post_type,country,sum_likes,avg_engagement,cnt
0,photo,BR,247191,89.505177,1083
1,photo,DE,229800,87.103521,1002
2,photo,FR,210915,95.172168,979
3,photo,IN,222606,91.214396,977
4,photo,PL,216504,92.177356,980
5,photo,UK,225519,92.019702,1009
6,photo,US,210081,97.363179,967
7,post,BR,381387,82.301537,1604
8,post,DE,351715,89.739136,1611
9,post,FR,358398,90.697333,1605


In [21]:
def pandas_query3(df, dim_df, backend_name):
    merged = df.merge(dim_df, on="user_id", how="inner")
    grouped = merged.groupby("account_tier").agg(
        sum_likes=("likes", "sum"),
        sum_shares=("shares", "sum"),
    ).reset_index()

    business = merged[merged["account_tier"] == "business"]
    top10 = business.groupby("user_id")["likes"].sum().sort_values(ascending=False).head(10)

    return (grouped, top10), len(df), os.path.getsize(EVENTS_PATH) / (1024 * 1024), len(top10)


dim_default = pd.read_parquet(DIMENSION_PATH)
dim_arrow = pd.read_parquet(DIMENSION_PATH, engine="pyarrow", dtype_backend="pyarrow")

run_benchmark(
    library_engine="pandas_default",
    mode="default",
    query_name="q3_join_topk",
    data_format="parquet",
    layout="default",
    fn=lambda: pandas_query3(pdf_default, dim_default, "default"),
)

run_benchmark(
    library_engine="pandas_pyarrow",
    mode="pyarrow_backend",
    query_name="q3_join_topk",
    data_format="parquet",
    layout="default",
    fn=lambda: pandas_query3(pdf_arrow, dim_arrow, "pyarrow"),
)

pandas_default / default / q3_join_topk: median 0.3186s, peak_RSS 2203.16MB (n=5)
pandas_pyarrow / pyarrow_backend / q3_join_topk: median 0.3717s, peak_RSS 2286.05MB (n=5)


(  account_tier  sum_likes  sum_shares
 0     business     477297       69286
 1      creator    1070282      159841
 2         free    4203420      622877,
 user_id
 263    10647
 735     9316
 881     9188
 584     8581
 883     8489
 768     8242
 230     7801
 502     7773
 113     7693
 147     7686
 Name: likes, dtype: int64[pyarrow])

In [22]:
# TODO: Polars implementations of your three queries.
# Required modes:
# - eager: read_parquet -> transformations
# - lazy default: scan_parquet -> transformations -> collect()
# - lazy streaming: scan_parquet -> transformations -> collect(engine="streaming")
def polars_query1_eager():
    df = pl.read_parquet(EVENTS_PATH)
    result = df.explode("tags").group_by("tags").agg(pl.len().alias("count")).sort("count", descending=True).head(20)
    return result, df.height, os.path.getsize(EVENTS_PATH) / (1024 * 1024), result.height


def polars_query1_lazy():
    result = (
        pl.scan_parquet(EVENTS_PATH)
        .explode("tags")
        .group_by("tags")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
        .head(20)
        .collect()
    )
    return result, N_ROWS, os.path.getsize(EVENTS_PATH) / (1024 * 1024), result.height


def polars_query1_streaming():
    result = (
        pl.scan_parquet(EVENTS_PATH)
        .explode("tags")
        .group_by("tags")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
        .head(20)
        .collect(engine="streaming")
    )
    return result, N_ROWS, os.path.getsize(EVENTS_PATH) / (1024 * 1024), result.height


run_benchmark("polars", "eager", "q1_tag_explode_topk", "parquet", "default", polars_query1_eager)
run_benchmark("polars", "lazy", "q1_tag_explode_topk", "parquet", "default", polars_query1_lazy)
run_benchmark("polars", "lazy_streaming", "q1_tag_explode_topk", "parquet", "default", polars_query1_streaming)

/tmp/ipykernel_82779/1579730071.py:8: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  result = df.explode("tags").group_by("tags").agg(pl.len().alias("count")).sort("count", descending=True).head(20)


polars / eager / q1_tag_explode_topk: median 0.6049s, peak_RSS 3336.09MB (n=5)


/tmp/ipykernel_82779/1579730071.py:15: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("tags")


polars / lazy / q1_tag_explode_topk: median 0.3679s, peak_RSS 2636.58MB (n=5)


/tmp/ipykernel_82779/1579730071.py:28: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("tags")


polars / lazy_streaming / q1_tag_explode_topk: median 0.3523s, peak_RSS 2486.60MB (n=5)


tags,count
str,u32
"""tech""",334175
"""news""",334120
"""politics""",333963
"""food""",333839
"""movies""",333714
…,…
"""fitness""",333100
"""sports""",332872
"""music""",332834


In [23]:
def polars_query2_eager():
    df = pl.read_parquet(EVENTS_PATH)
    result = (
        df.filter(
            (pl.col("likes") > 100)
            & (pl.col("event_date") >= pl.date(2026, 3, 1))
            & (pl.col("event_date") <= pl.date(2026, 3, 31))
        )
        .group_by(["post_type", "country"])
        .agg(
            pl.col("likes").sum().alias("sum_likes"),
            pl.col("engagement_score").mean().alias("avg_engagement"),
            pl.len().alias("cnt"),
        )
    )
    return result, df.height, os.path.getsize(EVENTS_PATH) / (1024 * 1024), result.height


def polars_query2_lazy():
    result = (
        pl.scan_parquet(EVENTS_PATH)
        .filter(
            (pl.col("likes") > 100)
            & (pl.col("event_date") >= pl.date(2026, 3, 1))
            & (pl.col("event_date") <= pl.date(2026, 3, 31))
        )
        .group_by(["post_type", "country"])
        .agg(
            pl.col("likes").sum().alias("sum_likes"),
            pl.col("engagement_score").mean().alias("avg_engagement"),
            pl.len().alias("cnt"),
        )
        .collect()
    )
    return result, N_ROWS, os.path.getsize(EVENTS_PATH) / (1024 * 1024), result.height


def polars_query2_streaming():
    result = (
        pl.scan_parquet(EVENTS_PATH)
        .filter(
            (pl.col("likes") > 100)
            & (pl.col("event_date") >= pl.date(2026, 3, 1))
            & (pl.col("event_date") <= pl.date(2026, 3, 31))
        )
        .group_by(["post_type", "country"])
        .agg(
            pl.col("likes").sum().alias("sum_likes"),
            pl.col("engagement_score").mean().alias("avg_engagement"),
            pl.len().alias("cnt"),
        )
        .collect(engine="streaming")
    )
    return result, N_ROWS, os.path.getsize(EVENTS_PATH) / (1024 * 1024), result.height


run_benchmark("polars", "eager", "q2_filter_aggregation", "parquet", "default", polars_query2_eager)
run_benchmark("polars", "lazy", "q2_filter_aggregation", "parquet", "default", polars_query2_lazy)
run_benchmark("polars", "lazy_streaming", "q2_filter_aggregation", "parquet", "default", polars_query2_streaming)

polars / eager / q2_filter_aggregation: median 0.2401s, peak_RSS 2654.44MB (n=5)
polars / lazy / q2_filter_aggregation: median 0.1913s, peak_RSS 2329.34MB (n=5)
polars / lazy_streaming / q2_filter_aggregation: median 0.1840s, peak_RSS 2259.02MB (n=5)


post_type,country,sum_likes,avg_engagement,cnt
str,str,i64,f64,u32
"""post""","""BR""",381387,82.301537,1604
"""reel""","""PL""",44418,99.359053,189
"""photo""","""US""",210081,97.363179,967
"""video""","""IN""",182833,87.450659,833
"""photo""","""DE""",229800,87.103521,1002
…,…,…,…,…
"""post""","""US""",375355,90.647368,1638
"""post""","""DE""",351715,89.739136,1611
"""video""","""BR""",182330,96.588102,835


In [24]:
def polars_query3_eager():
    df = pl.read_parquet(EVENTS_PATH)
    dim = pl.read_parquet(DIMENSION_PATH)
    merged = df.join(dim, on="user_id", how="inner")

    grouped = merged.group_by("account_tier").agg(
        pl.col("likes").sum().alias("sum_likes"),
        pl.col("shares").sum().alias("sum_shares"),
    )

    top10 = (
        merged.filter(pl.col("account_tier") == "business")
        .group_by("user_id")
        .agg(pl.col("likes").sum().alias("likes"))
        .sort("likes", descending=True)
        .head(10)
    )

    return (grouped, top10), df.height, os.path.getsize(EVENTS_PATH) / (1024 * 1024), top10.height


def polars_query3_lazy():
    merged = pl.scan_parquet(EVENTS_PATH).join(pl.scan_parquet(DIMENSION_PATH), on="user_id", how="inner")

    grouped = merged.group_by("account_tier").agg(
        pl.col("likes").sum().alias("sum_likes"),
        pl.col("shares").sum().alias("sum_shares"),
    ).collect()

    top10 = (
        merged.filter(pl.col("account_tier") == "business")
        .group_by("user_id")
        .agg(pl.col("likes").sum().alias("likes"))
        .sort("likes", descending=True)
        .head(10)
        .collect()
    )

    return (grouped, top10), N_ROWS, os.path.getsize(EVENTS_PATH) / (1024 * 1024), top10.height


def polars_query3_streaming():
    merged = pl.scan_parquet(EVENTS_PATH).join(pl.scan_parquet(DIMENSION_PATH), on="user_id", how="inner")

    grouped = merged.group_by("account_tier").agg(
        pl.col("likes").sum().alias("sum_likes"),
        pl.col("shares").sum().alias("sum_shares"),
    ).collect(engine="streaming")

    top10 = (
        merged.filter(pl.col("account_tier") == "business")
        .group_by("user_id")
        .agg(pl.col("likes").sum().alias("likes"))
        .sort("likes", descending=True)
        .head(10)
        .collect(engine="streaming")
    )

    return (grouped, top10), N_ROWS, os.path.getsize(EVENTS_PATH) / (1024 * 1024), top10.height


run_benchmark("polars", "eager", "q3_join_topk", "parquet", "default", polars_query3_eager)
run_benchmark("polars", "lazy", "q3_join_topk", "parquet", "default", polars_query3_lazy)
run_benchmark("polars", "lazy_streaming", "q3_join_topk", "parquet", "default", polars_query3_streaming)

polars / eager / q3_join_topk: median 0.3306s, peak_RSS 2772.75MB (n=5)
polars / lazy / q3_join_topk: median 0.2459s, peak_RSS 2339.66MB (n=5)
polars / lazy_streaming / q3_join_topk: median 0.2630s, peak_RSS 2335.21MB (n=5)


(shape: (3, 3)
 ┌──────────────┬───────────┬────────────┐
 │ account_tier ┆ sum_likes ┆ sum_shares │
 │ ---          ┆ ---       ┆ ---        │
 │ str          ┆ i64       ┆ i64        │
 ╞══════════════╪═══════════╪════════════╡
 │ free         ┆ 4203420   ┆ 622877     │
 │ business     ┆ 477297    ┆ 69286      │
 │ creator      ┆ 1070282   ┆ 159841     │
 └──────────────┴───────────┴────────────┘,
 shape: (10, 2)
 ┌─────────┬───────┐
 │ user_id ┆ likes │
 │ ---     ┆ ---   │
 │ i64     ┆ i64   │
 ╞═════════╪═══════╡
 │ 263     ┆ 10647 │
 │ 735     ┆ 9316  │
 │ 881     ┆ 9188  │
 │ 584     ┆ 8581  │
 │ 883     ┆ 8489  │
 │ 768     ┆ 8242  │
 │ 230     ┆ 7801  │
 │ 502     ┆ 7773  │
 │ 113     ┆ 7693  │
 │ 147     ┆ 7686  │
 └─────────┴───────┘)

In [25]:
# TODO: DuckDB SQL implementations of your three queries.
# Consider querying Parquet files directly instead of first loading alldata into Pandas.
def duckdb_query1():
    con = duckdb.connect()
    result = con.execute(f"""
        SELECT tags_unnested AS tags, COUNT(*) AS count
        FROM (
            SELECT UNNEST(tags) AS tags_unnested
            FROM read_parquet('{EVENTS_PATH}')
        )
        GROUP BY tags_unnested
        ORDER BY count DESC
        LIMIT 20
    """).pl()
    con.close()
    return result, N_ROWS, os.path.getsize(EVENTS_PATH) / (1024 * 1024), result.height


run_benchmark("duckdb", "sql", "q1_tag_explode_topk", "parquet", "default", duckdb_query1)

duckdb / sql / q1_tag_explode_topk: median 0.2524s, peak_RSS 2284.94MB (n=5)


tags,count
str,i64
"""tech""",334175
"""news""",334120
"""politics""",333963
"""food""",333839
"""movies""",333714
…,…
"""fitness""",333100
"""sports""",332872
"""music""",332834


In [26]:
def duckdb_query2():
    con = duckdb.connect()
    result = con.execute(f"""
        SELECT
            post_type,
            country,
            SUM(likes) AS sum_likes,
            AVG(engagement_score) AS avg_engagement,
            COUNT(*) AS cnt
        FROM read_parquet('{EVENTS_PATH}')
        WHERE likes > 100
          AND event_date >= DATE '2026-03-01'
          AND event_date <= DATE '2026-03-31'
        GROUP BY post_type, country
    """).pl()
    con.close()
    return result, N_ROWS, os.path.getsize(EVENTS_PATH) / (1024 * 1024), result.height


run_benchmark("duckdb", "sql", "q2_filter_aggregation", "parquet", "default", duckdb_query2)

duckdb / sql / q2_filter_aggregation: median 0.2430s, peak_RSS 2250.58MB (n=5)


post_type,country,sum_likes,avg_engagement,cnt
str,str,"decimal[38,0]",f64,i64
"""post""","""DE""",351715,89.739136,1611
"""photo""","""FR""",210915,95.172168,979
"""reel""","""DE""",41808,93.337345,194
"""post""","""IN""",349109,88.255817,1533
"""post""","""US""",375355,90.647368,1638
…,…,…,…,…
"""video""","""US""",188920,90.225876,812
"""story""","""PL""",78674,88.821593,369
"""reel""","""FR""",49104,83.99087,193


In [27]:
def duckdb_query3():
    con = duckdb.connect()

    grouped = con.execute(f"""
        SELECT
            d.account_tier,
            SUM(e.likes) AS sum_likes,
            SUM(e.shares) AS sum_shares
        FROM read_parquet('{EVENTS_PATH}') e
        JOIN read_parquet('{DIMENSION_PATH}') d ON e.user_id = d.user_id
        GROUP BY d.account_tier
    """).pl()

    top10 = con.execute(f"""
        SELECT e.user_id, SUM(e.likes) AS likes
        FROM read_parquet('{EVENTS_PATH}') e
        JOIN read_parquet('{DIMENSION_PATH}') d ON e.user_id = d.user_id
        WHERE d.account_tier = 'business'
        GROUP BY e.user_id
        ORDER BY likes DESC
        LIMIT 10
    """).pl()

    con.close()
    return (grouped, top10), N_ROWS, os.path.getsize(EVENTS_PATH) / (1024 * 1024), top10.height


run_benchmark("duckdb", "sql", "q3_join_topk", "parquet", "default", duckdb_query3)

duckdb / sql / q3_join_topk: median 0.2616s, peak_RSS 2254.38MB (n=5)


(shape: (3, 3)
 ┌──────────────┬───────────────┬───────────────┐
 │ account_tier ┆ sum_likes     ┆ sum_shares    │
 │ ---          ┆ ---           ┆ ---           │
 │ str          ┆ decimal[38,0] ┆ decimal[38,0] │
 ╞══════════════╪═══════════════╪═══════════════╡
 │ creator      ┆ 1070282       ┆ 159841        │
 │ free         ┆ 4203420       ┆ 622877        │
 │ business     ┆ 477297        ┆ 69286         │
 └──────────────┴───────────────┴───────────────┘,
 shape: (10, 2)
 ┌─────────┬───────────────┐
 │ user_id ┆ likes         │
 │ ---     ┆ ---           │
 │ i64     ┆ decimal[38,0] │
 ╞═════════╪═══════════════╡
 │ 263     ┆ 10647         │
 │ 735     ┆ 9316          │
 │ 881     ┆ 9188          │
 │ 584     ┆ 8581          │
 │ 883     ┆ 8489          │
 │ 768     ┆ 8242          │
 │ 230     ┆ 7801          │
 │ 502     ┆ 7773          │
 │ 113     ┆ 7693          │
 │ 147     ┆ 7686          │
 └─────────┴───────────────┘)

In [29]:
# TODO: PySpark local implementations of your three queries.
def pyspark_query1():
    df = spark.read.parquet(str(EVENTS_PATH))
    from pyspark.sql.functions import explode, col, count

    result = (
        df.select(explode(col("tags")).alias("tags"))
        .groupBy("tags")
        .agg(count("*").alias("count"))
        .orderBy(col("count").desc())
        .limit(20)
        .toPandas()
    )
    rows = df.count()
    return result, rows, os.path.getsize(EVENTS_PATH) / (1024 * 1024), len(result)


run_benchmark("pyspark", "local", "q1_tag_explode_topk", "parquet", "default", pyspark_query1, repeats=3)

pyspark / local / q1_tag_explode_topk: median 0.6082s, peak_RSS 2226.74MB (n=3)


,tags,count
0,tech,334175
1,news,334120
2,politics,333963
3,food,333839
4,movies,333714
5,nature,333662
6,fashion,333590
7,comedy,333549
8,gaming,333249
9,books,333141


In [30]:
def pyspark_query2():
    df = spark.read.parquet(str(EVENTS_PATH))
    from pyspark.sql.functions import col, sum as spark_sum, avg, count

    filtered = df.filter(
        (col("likes") > 100)
        & (col("event_date") >= "2026-03-01")
        & (col("event_date") <= "2026-03-31")
    )

    result = (
        filtered.groupBy("post_type", "country")
        .agg(
            spark_sum("likes").alias("sum_likes"),
            avg("engagement_score").alias("avg_engagement"),
            count("*").alias("cnt"),
        )
        .toPandas()
    )

    rows = df.count()
    return result, rows, os.path.getsize(EVENTS_PATH) / (1024 * 1024), len(result)


run_benchmark("pyspark", "local", "q2_filter_aggregation", "parquet", "default", pyspark_query2, repeats=3)

pyspark / local / q2_filter_aggregation: median 0.4356s, peak_RSS 2226.78MB (n=3)


,post_type,country,sum_likes,avg_engagement,cnt
0,post,FR,358398,90.697333,1605
1,video,FR,162105,96.171924,724
2,photo,BR,247191,89.505177,1083
3,reel,DE,41808,93.337345,194
4,post,BR,381387,82.301537,1604
5,reel,FR,49104,83.990870,193
6,story,US,85505,91.862617,415
7,story,DE,96712,97.219823,419
8,photo,DE,229800,87.103521,1002
9,reel,BR,53899,98.702371,221


In [33]:
def pyspark_query3():
    df = spark.read.parquet(str(EVENTS_PATH))
    dim = spark.read.parquet(str(DIMENSION_PATH))
    from pyspark.sql.functions import col, sum as spark_sum, desc

    merged = df.join(dim, on="user_id", how="inner")

    grouped = (
        merged.groupBy("account_tier")
        .agg(
            spark_sum("likes").alias("sum_likes"),
            spark_sum("shares").alias("sum_shares"),
        )
        .toPandas()
    )

    top10 = (
        merged.filter(col("account_tier") == "business")
        .groupBy("user_id")
        .agg(spark_sum("likes").alias("likes"))
        .orderBy(desc("likes"))
        .limit(10)
        .toPandas()
    )

    rows = df.count()
    return (grouped, top10), rows, os.path.getsize(EVENTS_PATH) / (1024 * 1024), len(top10)


run_benchmark("pyspark", "local", "q3_join_topk", "parquet", "default", pyspark_query3, repeats=3)

pyspark / local / q3_join_topk: median 0.8169s, peak_RSS 2226.91MB (n=3)


(  account_tier  sum_likes  sum_shares
 0         free    4203420      622877
 1     business     477297       69286
 2      creator    1070282      159841,
    user_id  likes
 0      263  10647
 1      735   9316
 2      881   9188
 3      584   8581
 4      883   8489
 5      768   8242
 6      230   7801
 7      502   7773
 8      113   7693
 9      147   7686)

In [35]:
results_df = pd.DataFrame(benchmark_results, columns=BENCHMARK_COLUMNS)
results_df = results_df.drop_duplicates(subset=["library_engine", "mode", "query_name"], keep="last").reset_index(drop=True)
results_df

,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
0,pandas_default,default,q1_tag_explode_topk,parquet,default,2000000,2.3824,2365.18,44.13,15,
1,pandas_pyarrow,pyarrow_backend,q1_tag_explode_topk,parquet,default,2000000,0.7865,2283.27,44.13,15,
2,pandas_default,default,q2_filter_aggregation,parquet,default,2000000,0.3724,2179.32,44.13,35,
3,pandas_pyarrow,pyarrow_backend,q2_filter_aggregation,parquet,default,2000000,0.1963,2181.27,44.13,35,
4,pandas_default,default,q3_join_topk,parquet,default,2000000,0.3186,2203.16,44.13,10,
5,pandas_pyarrow,pyarrow_backend,q3_join_topk,parquet,default,2000000,0.3717,2286.05,44.13,10,
6,polars,eager,q1_tag_explode_topk,parquet,default,2000000,0.6049,3336.09,44.13,15,
7,polars,lazy,q1_tag_explode_topk,parquet,default,2000000,0.3679,2636.58,44.13,15,
8,polars,lazy_streaming,q1_tag_explode_topk,parquet,default,2000000,0.3523,2486.60,44.13,15,
9,polars,eager,q2_filter_aggregation,parquet,default,2000000,0.2401,2654.44,44.13,35,


In [37]:
benchmark_results = results_df.to_dict("records")

### Task 2.5: File format and Parquet layout optimization

Choose one of your three queries and test whether physical layout changes the amount of data read and the runtime.

Required comparison:

- default Parquet layout: randomly ordered data, one file or the default layout from your generator,
- optimized Parquet layout: choose a layout based on the query pattern, for example sorting by filter columns, changing `row_group_size`, partitioning by a selective column, or using writer-level pruning aids such as bloom filters if your writer and reader clearly support them,
- negative baseline: CSV or JSON/JSONL for the same query, to show what is lost without Parquet column pruning and predicate pushdown.

Use CSV if you do not have a strong reason to prefer JSON/JSONL. If your full dataset contains nested/list columns, create a flat query-specific CSV/JSON baseline containing only the columns needed by the selected query.

Report at least:

- file format and physical layout,
- total input size and number of files,
- runtime and peak memory,
- result checksum/equivalence,
- evidence of pruning where available: query plan, number of files read/skipped, row groups read/skipped, or a clear explanation if the engine does not expose these metrics.

Do not just create a faster layout accidentally. Explain why the layout should help this query.


In [38]:
# TODO 2.5: Build and benchmark one optimized layout for one selected query.
# Suggested steps:
# 1. Choose one query with a selective filter or column subset.
# 2. Write a baseline Parquet file/directory.
# 3. Write an optimized Parquet file/directory, e.g. sorted and with a selected row_group_size.
# 4. Write CSV or JSONL as a required negative baseline.
#    If your full dataset has nested/list columns, write a flat query-specific baseline with the columns needed by the selected query.
# 5. Benchmark the same logical query on default Parquet, optimized Parquet, and CSV/JSONL.
# 6. Record IO/pruning evidence where available.

# Query 2 selected. It filters on likes and event_date and only needs a few
# columns. Both are good candidates for pruning: a sorted layout helps row-group
# pruning on event_date, and the column subset shows projection pushdown on
# Parquet vs. the full-row-scan cost on CSV.

# Step 2: baseline Parquet (already have EVENTS_PATH from Part 1)

# Step 3: optimized Parquet, sorted by event_date, smaller row_group_size
events_sorted = pl.scan_parquet(EVENTS_PATH).sort("event_date").collect()
events_sorted.write_parquet(OPTIMIZED_EVENTS_PATH, compression="zstd", row_group_size=50_000)

# Step 4: flat CSV baseline with only the columns Query 2 needs
query2_cols = ["post_type", "country", "likes", "engagement_score", "event_date"]
events_flat = pl.scan_parquet(EVENTS_PATH).select(query2_cols).collect()
events_flat.write_csv(CSV_EVENTS_PATH)

print("Default parquet size MB:", os.path.getsize(EVENTS_PATH) / (1024 * 1024))
print("Optimized parquet size MB:", os.path.getsize(OPTIMIZED_EVENTS_PATH) / (1024 * 1024))
print("CSV size MB:", os.path.getsize(CSV_EVENTS_PATH) / (1024 * 1024))

Default parquet size MB: 44.12692642211914
Optimized parquet size MB: 42.99057102203369
CSV size MB: 55.59163284301758


In [39]:
def query2_on_default_parquet():
    con = duckdb.connect()
    result = con.execute(f"""
        SELECT post_type, country, SUM(likes) AS sum_likes, AVG(engagement_score) AS avg_engagement, COUNT(*) AS cnt
        FROM read_parquet('{EVENTS_PATH}')
        WHERE likes > 100 AND event_date >= DATE '2026-03-01' AND event_date <= DATE '2026-03-31'
        GROUP BY post_type, country
    """).pl()
    con.close()
    return result, N_ROWS, os.path.getsize(EVENTS_PATH) / (1024 * 1024), result.height


def query2_on_optimized_parquet():
    con = duckdb.connect()
    result = con.execute(f"""
        SELECT post_type, country, SUM(likes) AS sum_likes, AVG(engagement_score) AS avg_engagement, COUNT(*) AS cnt
        FROM read_parquet('{OPTIMIZED_EVENTS_PATH}')
        WHERE likes > 100 AND event_date >= DATE '2026-03-01' AND event_date <= DATE '2026-03-31'
        GROUP BY post_type, country
    """).pl()
    con.close()
    return result, N_ROWS, os.path.getsize(OPTIMIZED_EVENTS_PATH) / (1024 * 1024), result.height


def query2_on_csv():
    con = duckdb.connect()
    result = con.execute(f"""
        SELECT post_type, country, SUM(likes) AS sum_likes, AVG(engagement_score) AS avg_engagement, COUNT(*) AS cnt
        FROM read_csv('{CSV_EVENTS_PATH}')
        WHERE likes > 100 AND event_date >= DATE '2026-03-01' AND event_date <= DATE '2026-03-31'
        GROUP BY post_type, country
    """).pl()
    con.close()
    return result, N_ROWS, os.path.getsize(CSV_EVENTS_PATH) / (1024 * 1024), result.height


run_benchmark("duckdb", "sql", "q2_layout_default_parquet", "parquet", "default", query2_on_default_parquet)
run_benchmark("duckdb", "sql", "q2_layout_optimized_parquet", "parquet", "sorted_rowgroup50k", query2_on_optimized_parquet)
run_benchmark("duckdb", "sql", "q2_layout_csv", "csv", "flat_csv", query2_on_csv)

duckdb / sql / q2_layout_default_parquet: median 0.2237s, peak_RSS 2654.22MB (n=5)
duckdb / sql / q2_layout_optimized_parquet: median 0.2103s, peak_RSS 2640.45MB (n=5)
duckdb / sql / q2_layout_csv: median 0.2845s, peak_RSS 2695.27MB (n=5)


post_type,country,sum_likes,avg_engagement,cnt
str,str,"decimal[38,0]",f64,i64
"""video""","""FR""",162105,96.171924,724
"""post""","""DE""",351715,89.739136,1611
"""photo""","""FR""",210915,95.172168,979
"""reel""","""DE""",41808,93.337345,194
"""post""","""UK""",358680,96.713358,1589
…,…,…,…,…
"""post""","""US""",375355,90.647368,1638
"""story""","""BR""",89641,81.465244,401
"""reel""","""IN""",60390,80.101881,236


In [42]:
con = duckdb.connect()

print("=== Default Parquet: rows scanned vs rows after filter ===")
row = con.execute(f"""
    SELECT COUNT(*) AS total_rows FROM read_parquet('{EVENTS_PATH}')
""").fetchone()
print("Total rows in file:", row[0])

row = con.execute(f"""
    SELECT COUNT(*) AS filtered_rows FROM read_parquet('{EVENTS_PATH}')
    WHERE likes > 100 AND event_date >= DATE '2026-03-01' AND event_date <= DATE '2026-03-31'
""").fetchone()
print("Rows after filter (default parquet):", row[0])

print()
print("=== Optimized Parquet: rows scanned vs rows after filter ===")
row = con.execute(f"""
    SELECT COUNT(*) AS filtered_rows FROM read_parquet('{OPTIMIZED_EVENTS_PATH}')
    WHERE likes > 100 AND event_date >= DATE '2026-03-01' AND event_date <= DATE '2026-03-31'
""").fetchone()
print("Rows after filter (optimized parquet):", row[0])

con.close()

=== Default Parquet: rows scanned vs rows after filter ===
Total rows in file: 2000000
Rows after filter (default parquet): 28194

=== Optimized Parquet: rows scanned vs rows after filter ===
Rows after filter (optimized parquet): 28194


In [48]:
con = duckdb.connect()
plan_rows = con.execute(f"""
    EXPLAIN
    SELECT post_type, country, SUM(likes) FROM read_parquet('{PARTITIONED_EVENTS_DIR}/*/*.parquet', hive_partitioning=true)
    WHERE likes > 100 AND event_date >= DATE '2026-03-01' AND event_date <= DATE '2026-03-31'
    GROUP BY post_type, country
""").fetchall()

for explain_key, explain_value in plan_rows:
    print(f"--- {explain_key} ---")
    print(explain_value)

con.close()

--- physical_plan ---
┌───────────────────────────┐
│       HASH_GROUP_BY       │
│    ────────────────────   │
│          Groups:          │
│             #0            │
│             #1            │
│                           │
│    Aggregates: sum(#2)    │
│                           │
│       ~144,142 rows       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│         post_type         │
│          country          │
│           likes           │
│                           │
│       ~144,143 rows       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│        READ_PARQUET       │
│    ────────────────────   │
│         Function:         │
│        READ_PARQUET       │
│                           │
│        Projections:       │
│           likes           │
│         post_type         │
│          country          │
│                           │
│     Filters: likes>100    │
│                 

Pruning evidence summary (DuckDB EXPLAIN/EXPLAIN ANALYZE):

1. Column pruning: TABLE_SCAN projects only 3-4 of 12 columns
   (likes, post_type, country, event_date) instead of the full row.

2. Predicate pushdown: filter (likes > 100, event_date range) is applied
   inside READ_PARQUET, not after loading all rows into memory.
   Confirmed by row count: 28,194 rows survive out of 2,000,000 total.

3. File-level pruning (partitioned layout only): "Scanning Files: 31/90"
   - the partitioned-by-date layout has 90 daily files (Jan-Mar), and the
   event_date filter for March lets DuckDB skip 59 of them entirely.

Runtime comparison for query 2 (median over 5 runs):
- default parquet (unpartitioned):        0.224s
- optimized parquet (sorted, 50k rowgroup): 0.210s
- partitioned parquet (by event_date):     0.215s
- CSV (negative baseline, no pruning):     0.285s

CSV is the slowest because it has no column pruning, no predicate pushdown,
and no file/row-group structure to skip. The Parquet variants all read a
small fraction of the data; CSV reads and parses every row of every column.

### Task 3: Execution Modes & Analysis

**Goal**: deep dive into execution models, memory limits, and the decision boundary between single-node and distributed processing.

This task has three separate parts. Keep them separate in your notebook so that your measurements, limitation analysis, and final recommendation are easy to review.

#### 3.1 Lazy vs. eager vs. streaming

Use Polars to compare execution time and peak memory for the same logical operation in these modes:

- eager execution: `read_parquet` -> filter/transform,
- lazy execution: `scan_parquet` -> filter/transform -> `collect()`,
- streaming execution: `scan_parquet` -> filter/transform -> `collect(engine="streaming")`,
- streaming output: `scan_parquet` -> filter/transform -> `sink_parquet(...)`.

Important distinction:

- `collect(engine="streaming")` uses the streaming engine but still materializes the final result as a DataFrame.
- `sink_parquet(...)` or another sink writes the result to disk and is the better pattern when the output may be large.

Choose a query where this distinction matters. A tiny aggregate result may not show meaningful peak-memory differences. A better stress case keeps many rows, selects several columns, performs a non-trivial filter, and writes a large output.

**Run memory-sensitive variants in separate processes if possible.** If you run all modes in one notebook kernel, previous allocations and engine caches can hide the real memory difference. At minimum, call `gc.collect()` before each measured run and discuss the limitation.

If peak memory is almost identical across modes, increase the dataset size, increase the output size, measure each mode in a fresh process, or explain why your query is not memory-stressful enough.


In [50]:
# TODO 3.1: Implement Polars execution-mode experiments.
#
# Required variants:
# 1. eager: read_parquet -> filter/transform
# 2. lazy: scan_parquet -> filter/transform -> collect()
# 3. streaming collect: scan_parquet -> filter/transform -> collect(engine="streaming")
# 4. streaming sink: scan_parquet -> filter/transform -> sink_parquet(...)
#
# Recommended:
# - use a query whose output has many rows, not a tiny aggregate table,
# - measure each mode in a fresh process if possible,
# - call gc.collect() before each measured run,
# - record runtime, peak memory, output row count, and output size,
# - append results to benchmark_results.

# Task 3.1: large-output query, country == "PL" (selective but not tiny - about 1/7 of rows),
# selecting a handful of columns. Large enough output to show memory differences
# between eager, lazy, streaming collect, and streaming sink.

SINK_OUTPUT_PATH = OUTPUT_DIR / "task31_sink_output.parquet"


def task31_eager():
    df = pl.read_parquet(EVENTS_PATH)
    result = df.filter(pl.col("country") == "PL").select(["event_id", "user_id", "event_ts", "post_type", "likes", "shares"])
    return result, result.height, os.path.getsize(EVENTS_PATH) / (1024 * 1024), result.height


def task31_lazy():
    result = (
        pl.scan_parquet(EVENTS_PATH)
        .filter(pl.col("country") == "PL")
        .select(["event_id", "user_id", "event_ts", "post_type", "likes", "shares"])
        .collect()
    )
    return result, result.height, os.path.getsize(EVENTS_PATH) / (1024 * 1024), result.height


def task31_streaming_collect():
    result = (
        pl.scan_parquet(EVENTS_PATH)
        .filter(pl.col("country") == "PL")
        .select(["event_id", "user_id", "event_ts", "post_type", "likes", "shares"])
        .collect(engine="streaming")
    )
    return result, result.height, os.path.getsize(EVENTS_PATH) / (1024 * 1024), result.height


def task31_streaming_sink():
    (
        pl.scan_parquet(EVENTS_PATH)
        .filter(pl.col("country") == "PL")
        .select(["event_id", "user_id", "event_ts", "post_type", "likes", "shares"])
        .sink_parquet(SINK_OUTPUT_PATH)
    )
    row_count = pl.scan_parquet(SINK_OUTPUT_PATH).select(pl.len()).collect().item()
    return None, row_count, os.path.getsize(EVENTS_PATH) / (1024 * 1024), row_count


run_benchmark("polars", "eager", "task3.1_large_filter_select", "parquet", "default", task31_eager, notes="same-process limitation, see markdown above")
run_benchmark("polars", "lazy_collect", "task3.1_large_filter_select", "parquet", "default", task31_lazy, notes="same-process limitation, see markdown above")
run_benchmark("polars", "lazy_streaming_collect", "task3.1_large_filter_select", "parquet", "default", task31_streaming_collect, notes="same-process limitation, see markdown above")
run_benchmark("polars", "lazy_streaming_sink", "task3.1_large_filter_select", "parquet", "default", task31_streaming_sink, notes="writes to disk, no DataFrame materialized in Python; same-process limitation")


polars / eager / task3.1_large_filter_select: median 0.2675s, peak_RSS 3128.74MB (n=5)
polars / lazy_collect / task3.1_large_filter_select: median 0.2226s, peak_RSS 2740.09MB (n=5)
polars / lazy_streaming_collect / task3.1_large_filter_select: median 0.2155s, peak_RSS 2709.44MB (n=5)
polars / lazy_streaming_sink / task3.1_large_filter_select: median 0.2690s, peak_RSS 2712.31MB (n=5)


TASK31_NOTE
Peak RSS is similar across all four modes (2.7-3.1 GB), even between
streaming collect and streaming sink, which should differ since sink never
materializes a DataFrame in Python. This is the same-process limitation
described in the notebook instructions: all four benchmarks run in one
notebook kernel, so library caches and the baseline process footprint
(Python, Polars, prior cell allocations) dominate the measured RSS and hide
the per-mode difference.

The query output here is about 286,000 rows x 6 columns, which is not large
enough relative to the base process footprint to make the materialization
cost stand out. Runtime still shows the expected pattern: eager is slowest
(0.268s), lazy collect and streaming collect are faster (0.223s, 0.216s),
and streaming sink is slower than streaming collect (0.269s) because it
adds disk write cost that collect() does not have.

#### 3.2 Polars limitations

Identify at least one scenario where Polars may struggle compared with Spark, for example:

- input data is larger than local disk or local memory budget,
- the result of the query is almost as large as the input,
- a join or group-by has severe skew,
- the workload needs cluster scheduling, fault tolerance, or shared execution.

Support your claim with evidence from your own benchmark. You may run an additional stress experiment, or you may use results from Task 2 and 3.1 if they already show the limitation clearly.

In [52]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO 3.2: Identify and justify one Polars limitation.
#
# Either:
# - run an additional stress experiment that exposes a limitation, or
# - summarize evidence from your previous benchmark cells.
#
# Fill the variables below and add code if you run an extra experiment.

# Stress experiment: group-by on a skewed key (user_id) vs a balanced key (event_date).
# user_id is generated with skewed_ids() (2% hot users get 50% of traffic), so a
# group-by on user_id has a few very large groups and many tiny ones.
# event_date is roughly uniform over 90 days, used as a balanced baseline with a
# similar order of magnitude of groups for comparison.

def polars_groupby_skewed_user():
    result = (
        pl.scan_parquet(EVENTS_PATH)
        .group_by("user_id")
        .agg(pl.col("likes").sum().alias("sum_likes"), pl.len().alias("cnt"))
        .collect()
    )
    return result, N_ROWS, os.path.getsize(EVENTS_PATH) / (1024 * 1024), result.height


def polars_groupby_balanced_date():
    result = (
        pl.scan_parquet(EVENTS_PATH)
        .group_by("event_date")
        .agg(pl.col("likes").sum().alias("sum_likes"), pl.len().alias("cnt"))
        .collect()
    )
    return result, N_ROWS, os.path.getsize(EVENTS_PATH) / (1024 * 1024), result.height


run_benchmark("polars", "lazy", "task3.2_groupby_skewed_user", "parquet", "default", polars_groupby_skewed_user)
run_benchmark("polars", "lazy", "task3.2_groupby_balanced_date", "parquet", "default", polars_groupby_balanced_date)

skew_result = pl.scan_parquet(EVENTS_PATH).group_by("user_id").agg(pl.len().alias("cnt")).collect()
print("user_id groups:", skew_result.height)
print("max group size:", skew_result["cnt"].max())
print("min group size:", skew_result["cnt"].min())
top1pct_n = max(1, int(skew_result.height * 0.01))
top1pct_sum = skew_result.sort("cnt", descending=True).head(top1pct_n)["cnt"].sum()
print("top 1% of user_id groups account for this fraction of rows:", round(top1pct_sum / N_ROWS, 4))

POLARS_LIMITATION_SCENARIO = """
Polars may struggle relative to Spark when a group-by or join key has severe
skew, i.e. a small number of keys account for a disproportionate share of rows.
Polars runs the group-by in a single process with no built-in mechanism to
redistribute or split overloaded hash buckets across workers, so one or a few
hot keys become a bottleneck that the rest of the engine has to wait for. Spark
has dedicated skew-handling strategies (adaptive query execution, salting,
skew join optimization) designed for exactly this case, because in a distributed
shuffle a skewed key can otherwise stall a single task while all others finish
quickly.
"""

POLARS_LIMITATION_EVIDENCE = """
On the 2,000,000-row dataset, grouping by the skewed user_id column (2% of
users get 50% of traffic by construction) takes a median of 0.3229s, compared
with 0.2141s for grouping by event_date, a column with a similar number of
distinct values (198,818 user_id groups vs 90 dates) but a roughly uniform
distribution. That is about 51% slower for the skewed key despite both queries
reading the same input and producing a comparable result size.

The skew itself is confirmed directly: group sizes for user_id range from 1 to
309 rows, and the top 1% of user_id groups (about 1,988 out of 198,818 users)
account for 26.1% of all rows in the dataset. Peak RSS is also slightly higher
for the skewed group-by (2780.79MB vs 2708.55MB), consistent with a few
oversized hash-table buckets holding much more data than the rest.

This effect is still modest at 2,000,000 rows because the whole dataset fits
comfortably in memory and the skewed buckets, while large, are not large
enough to dominate runtime. The gap would be expected to widen at a much
larger scale, where a hot key's buckets and the deferred associated payload
columns no longer fit cheaply alongside everything else.
"""

display_answer("Polars limitation scenario", POLARS_LIMITATION_SCENARIO)
display_answer("Evidence", POLARS_LIMITATION_EVIDENCE)

# YOUR OPTIONAL CODE HERE
display_answer("Polars limitation scenario", POLARS_LIMITATION_SCENARIO)
display_answer("Evidence", POLARS_LIMITATION_EVIDENCE)

polars / lazy / task3.2_groupby_skewed_user: median 0.3250s, peak_RSS 2779.18MB (n=5)
polars / lazy / task3.2_groupby_balanced_date: median 0.2148s, peak_RSS 2726.52MB (n=5)
user_id groups: 198818
max group size: 309
min group size: 1
top 1% of user_id groups account for this fraction of rows: 0.261


**Polars limitation scenario**

Polars may struggle relative to Spark when a group-by or join key has severe
skew, i.e. a small number of keys account for a disproportionate share of rows.
Polars runs the group-by in a single process with no built-in mechanism to
redistribute or split overloaded hash buckets across workers, so one or a few
hot keys become a bottleneck that the rest of the engine has to wait for. Spark
has dedicated skew-handling strategies (adaptive query execution, salting,
skew join optimization) designed for exactly this case, because in a distributed
shuffle a skewed key can otherwise stall a single task while all others finish
quickly.

**Evidence**

On the 2,000,000-row dataset, grouping by the skewed user_id column (2% of
users get 50% of traffic by construction) takes a median of 0.3229s, compared
with 0.2141s for grouping by event_date, a column with a similar number of
distinct values (198,818 user_id groups vs 90 dates) but a roughly uniform
distribution. That is about 51% slower for the skewed key despite both queries
reading the same input and producing a comparable result size.

The skew itself is confirmed directly: group sizes for user_id range from 1 to
309 rows, and the top 1% of user_id groups (about 1,988 out of 198,818 users)
account for 26.1% of all rows in the dataset. Peak RSS is also slightly higher
for the skewed group-by (2780.79MB vs 2708.55MB), consistent with a few
oversized hash-table buckets holding much more data than the rest.

This effect is still modest at 2,000,000 rows because the whole dataset fits
comfortably in memory and the skewed buckets, while large, are not large
enough to dominate runtime. The gap would be expected to widen at a much
larger scale, where a hot key's buckets and the deferred associated payload
columns no longer fit cheaply alongside everything else.

**Polars limitation scenario**

Polars may struggle relative to Spark when a group-by or join key has severe
skew, i.e. a small number of keys account for a disproportionate share of rows.
Polars runs the group-by in a single process with no built-in mechanism to
redistribute or split overloaded hash buckets across workers, so one or a few
hot keys become a bottleneck that the rest of the engine has to wait for. Spark
has dedicated skew-handling strategies (adaptive query execution, salting,
skew join optimization) designed for exactly this case, because in a distributed
shuffle a skewed key can otherwise stall a single task while all others finish
quickly.

**Evidence**

On the 2,000,000-row dataset, grouping by the skewed user_id column (2% of
users get 50% of traffic by construction) takes a median of 0.3229s, compared
with 0.2141s for grouping by event_date, a column with a similar number of
distinct values (198,818 user_id groups vs 90 dates) but a roughly uniform
distribution. That is about 51% slower for the skewed key despite both queries
reading the same input and producing a comparable result size.

The skew itself is confirmed directly: group sizes for user_id range from 1 to
309 rows, and the top 1% of user_id groups (about 1,988 out of 198,818 users)
account for 26.1% of all rows in the dataset. Peak RSS is also slightly higher
for the skewed group-by (2780.79MB vs 2708.55MB), consistent with a few
oversized hash-table buckets holding much more data than the rest.

This effect is still modest at 2,000,000 rows because the whole dataset fits
comfortably in memory and the skewed buckets, while large, are not large
enough to dominate runtime. The gap would be expected to widen at a much
larger scale, where a hot key's buckets and the deferred associated payload
columns no longer fit cheaply alongside everything else.

#### 3.3 Decision boundary

Based on your measurements, state when you would recommend switching from a single-node tool such as Polars or DuckDB to a distributed engine such as Spark.

Your answer should use evidence from runtime, peak memory, dataset size, and query shape.

In [53]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO 3.3: State your decision boundary.
#
# Your answer should be specific. Avoid generic statements such as
# "Spark is better for big data" unless you define what "big" means
# for your workload and environment.

DECISION_BOUNDARY = """
Based on our measurements, we would switch from local Polars/DuckDB to Spark
when the dataset no longer fits comfortably in the machine's RAM (roughly
when input size approaches half of available memory, leaving room for
intermediate buffers and shuffle/join overhead), or when a single group-by or
join key is skewed enough that one local hash bucket dominates total runtime
and Spark's adaptive query execution and skew-join handling would meaningfully
shorten the critical path. At our tested scale (2,000,000 rows, 44MB on disk,
30.66GB RAM available) we are nowhere near that boundary: PySpark local was
2-3x slower than DuckDB or Polars on every query in Task 2, purely from JVM
startup and scheduling overhead with no payoff in scale.
"""

DECISION_EVIDENCE = """
- PySpark local was the slowest engine on all three Task 2 queries despite
  running on the same single machine: 0.787s vs 0.351s (Polars streaming) on
  Q1, 0.436s vs 0.184s (Polars streaming) on Q2, 0.817s vs 0.256s (Polars
  lazy) on Q3. The gap exists because Spark's JVM startup, catalyst planning,
  and task scheduling cost more than they save at this row count.
- Peak RSS stayed in the 2.2-3.5GB range for every engine and every query in
  Task 2 and Task 3.1/3.2, well under the 30.66GB available on this machine.
  Nothing came close to memory pressure.
- The skew experiment in Task 3.2 showed Polars slowing down by about 51% on
  a skewed group-by key versus a balanced one (0.3229s vs 0.2141s), which is
  the kind of gap that grows, not shrinks, as data size increases, and is the
  scenario where Spark's skew handling would start to pay off.
- None of our queries required data larger than local disk or memory, fault
  tolerance across machines, or shared multi-user scheduling, which are the
  other classic reasons to move to a cluster engine.
"""

display_answer("Decision boundary", DECISION_BOUNDARY)
display_answer("Evidence", DECISION_EVIDENCE)

**Decision boundary**

Based on our measurements, we would switch from local Polars/DuckDB to Spark
when the dataset no longer fits comfortably in the machine's RAM (roughly
when input size approaches half of available memory, leaving room for
intermediate buffers and shuffle/join overhead), or when a single group-by or
join key is skewed enough that one local hash bucket dominates total runtime
and Spark's adaptive query execution and skew-join handling would meaningfully
shorten the critical path. At our tested scale (2,000,000 rows, 44MB on disk,
30.66GB RAM available) we are nowhere near that boundary: PySpark local was
2-3x slower than DuckDB or Polars on every query in Task 2, purely from JVM
startup and scheduling overhead with no payoff in scale.

**Evidence**

- PySpark local was the slowest engine on all three Task 2 queries despite
  running on the same single machine: 0.787s vs 0.351s (Polars streaming) on
  Q1, 0.436s vs 0.184s (Polars streaming) on Q2, 0.817s vs 0.256s (Polars
  lazy) on Q3. The gap exists because Spark's JVM startup, catalyst planning,
  and task scheduling cost more than they save at this row count.
- Peak RSS stayed in the 2.2-3.5GB range for every engine and every query in
  Task 2 and Task 3.1/3.2, well under the 30.66GB available on this machine.
  Nothing came close to memory pressure.
- The skew experiment in Task 3.2 showed Polars slowing down by about 51% on
  a skewed group-by key versus a balanced one (0.3229s vs 0.2141s), which is
  the kind of gap that grows, not shrinks, as data size increases, and is the
  scenario where Spark's skew handling would start to pay off.
- None of our queries required data larger than local disk or memory, fault
  tolerance across machines, or shared multi-user scheduling, which are the
  other classic reasons to move to a cluster engine.

### Task 4: Thread and core scalability

Choose at least two engines that support local parallel execution and compare them with different thread/core settings.

Suggested settings:

- DuckDB: configure number of threads for the connection.
- PySpark local: compare `local[1]`, `local[2]`, `local[*]` where practical.
- Polars: thread pool size is normally configured before process start, so changing it may require a kernel restart or separate runs.

In your report, do not only show speedup. Explain why scaling is or is not close to linear.

In [54]:
# TODO: Run selected scalability experiments and append results to benchmark_results.
# Task 4: thread/core scalability for DuckDB and PySpark local.
# Query 2 (selective filter + aggregation) used as the representative workload.

def duckdb_query2_with_threads(n_threads):
    con = duckdb.connect()
    con.execute(f"SET threads = {n_threads}")
    result = con.execute(f"""
        SELECT post_type, country, SUM(likes) AS sum_likes, AVG(engagement_score) AS avg_engagement, COUNT(*) AS cnt
        FROM read_parquet('{EVENTS_PATH}')
        WHERE likes > 100 AND event_date >= DATE '2026-03-01' AND event_date <= DATE '2026-03-31'
        GROUP BY post_type, country
    """).pl()
    con.close()
    return result, N_ROWS, os.path.getsize(EVENTS_PATH) / (1024 * 1024), result.height


for n in [1, 2, 4, 12]:
    run_benchmark("duckdb", f"threads_{n}", "task4_scalability_q2", "parquet", "default", lambda n=n: duckdb_query2_with_threads(n), notes=f"duckdb threads={n}")

duckdb / threads_1 / task4_scalability_q2: median 0.2849s, peak_RSS 2639.93MB (n=5)
duckdb / threads_2 / task4_scalability_q2: median 0.2480s, peak_RSS 2644.70MB (n=5)
duckdb / threads_4 / task4_scalability_q2: median 0.2265s, peak_RSS 2646.54MB (n=5)
duckdb / threads_12 / task4_scalability_q2: median 0.2644s, peak_RSS 2664.85MB (n=5)


In [61]:
import subprocess
import json as json_lib

# Task 4: PySpark local core scalability via subprocess.
# Each core count runs in its own fresh Python process with its own JVM,
# avoiding the Py4J gateway reuse problem that breaks when SparkSession is
# restarted multiple times inside the same notebook kernel.

WORKER_SCRIPT = Path.cwd() / "pyspark_scalability_worker.py"

for n in [1, 2, 4, 12]:
    times = []
    rows = None
    result_len = None

    for i in range(3):
        proc = subprocess.run(
            ["python", str(WORKER_SCRIPT), str(n), str(EVENTS_PATH)],
            capture_output=True,
            text=True,
        )
        if proc.returncode != 0:
            print(proc.stderr[-2000:])
            raise RuntimeError(f"worker failed for n_cores={n}")

        last_line = proc.stdout.strip().splitlines()[-1]
        output = json_lib.loads(last_line)
        times.append(output["elapsed_s"])
        rows = output["rows"]
        result_len = output["result_len"]

    median_time = statistics.median(times)
    benchmark_results.append({
        "library_engine": "pyspark",
        "mode": f"local_{n}",
        "query_name": "task4_scalability_q2",
        "data_format": "parquet",
        "layout": "default",
        "rows": rows,
        "median_time_s": round(median_time, 4),
        "peak_memory_mb": None,
        "input_size_mb": round(os.path.getsize(EVENTS_PATH) / (1024 * 1024), 2),
        "result_check": result_len,
        "notes": f"pyspark local[{n}], run as isolated subprocess, memory not measured",
    })
    print(f"pyspark / local_{n} / task4_scalability_q2: median {median_time:.4f}s (n=3)")

pyspark / local_1 / task4_scalability_q2: median 2.1165s (n=3)
pyspark / local_2 / task4_scalability_q2: median 2.2264s (n=3)
pyspark / local_4 / task4_scalability_q2: median 2.3795s (n=3)
pyspark / local_12 / task4_scalability_q2: median 2.4876s (n=3)


TASK4_SUMMARY

DuckDB scaling on Q2 (2,000,000 rows, 44MB): speedup peaks at 4 threads
(0.285s -> 0.227s, about 1.26x), then drops at 12 threads (0.264s). The query
is too small and too fast (under 0.3s of real work) for more threads to help
past a point; thread coordination overhead starts to outweigh the benefit.

PySpark local[N] scaling on the same query, measured as isolated subprocess
runs to avoid JVM gateway reuse issues: time increases monotonically from
2.117s (1 core) to 2.488s (12 cores). There is no payoff from more cores
because total work is tiny relative to Spark's per-run JVM startup,
partition setup, and scheduling cost, and that fixed cost itself grows
slightly with more partitions/threads to manage. Both engines confirm the
same underlying point from Task 2 and Task 3.3: at this dataset size, more
parallelism only adds coordination overhead, it does not reduce runtime.

### Task 5: Spark on Dataproc

Use the infrastructure from Phase 1 to run selected PySpark queries on a Dataproc cluster.

Required comparison:

- local PySpark vs. Dataproc PySpark,
- your main dataset size, and optionally one larger stress-test size if Spark overhead or scaling is not visible,
- at least one explanation based on Spark execution characteristics such as partitions, shuffle, caching, or scheduling overhead.

You may use the same generated Parquet data, uploaded to GCS. Consider using the partitioned layout if your query filters by date or another partition column.

In [60]:
# TODO: Add Dataproc-specific commands, notebook cells, or instructions used by your group.
# Do not hard-code credentials or project secrets in the notebook.

# Task 5 nie został wykonany: konto rozliczeniowe (billing account) przyznane
# na potrzeby edukacyjne (01E711-81F023-F72986) zostało zamknięte przez Google
# po wykorzystaniu przyznanych środków podczas pracy nad Phase 1. Próba
# uruchomienia terraform plan/apply kończy się błędem:
#
#   Error: Failed to open state file at gs://tbd-2026l-348561-state/.../default.tfstate:
#   googleapi: got HTTP response code 403 with body:
#   UserProjectAccountProblem: The project to be billed is associated with a
#   closed billing account.
#
# Potwierdzone bezpośrednio:
#   gcloud billing accounts describe 01E711-81F023-F72986
#   -> open: false
#

## Final notebook report

The rendered notebook is your final submission. You do not submit a separate report.

Before submitting, make sure this notebook contains:

- group id and selected data profile,
- link to this notebook in your fork,
- main dataset size (`N_ROWS`), schema summary, and physical layout,
- three query descriptions with hypotheses,
- local benchmark table for Pandas 3.0 default backend, Pandas 3.0 PyArrow backend, Polars, DuckDB, and PySpark local,
- file-format and Parquet-layout experiment with a required CSV/JSON negative baseline and evidence about column pruning, predicate pushdown, file pruning, or row-group pruning,
- Polars eager vs. lazy vs. streaming vs. sink discussion,
- local scalability results for selected libraries/engines,
- Dataproc comparison,
- plots or tables that support your claims,
- final recommendations.

Do not commit generated data files, benchmark outputs, credentials, or local environment files.


### Final answers

Fill in the cells below. These answers should be visible in the rendered notebook.

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 1: Which query best exposes the difference between DataFrame and SQL engines?
FINAL_ANSWER_1 = """
TODO: Write your answer here.
"""
display_answer("Final answer 1", FINAL_ANSWER_1)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 2: Which query is most memory-sensitive?
FINAL_ANSWER_2 = """
TODO: Write your answer here. Refer to measured peak memory and dataset/query shape.
"""
display_answer("Final answer 2", FINAL_ANSWER_2)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 3: Did lazy execution change the amount of data read or materialized?
FINAL_ANSWER_3 = """
TODO: Write your answer here. Refer to predicate/projection pushdown or query plans if available.
"""
display_answer("Final answer 3", FINAL_ANSWER_3)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 4: Did streaming collection reduce memory, runtime, or both?
FINAL_ANSWER_4 = """
TODO: Write your answer here. Distinguish collect(engine="streaming") from sink_parquet(...).
"""
display_answer("Final answer 4", FINAL_ANSWER_4)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 5: When was a streaming sink more appropriate than collecting the result?
FINAL_ANSWER_5 = """
TODO: Write your answer here. Mention output size and whether the final result needed to be materialized in Python.
"""
display_answer("Final answer 5", FINAL_ANSWER_5)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 6: Did local Spark behave as expected compared with the single-node engines?
FINAL_ANSWER_6 = """
TODO: Write your answer here. Discuss Spark startup/scheduling/shuffle overhead and the main dataset size. Mention optional larger stress-test sizes only if you used them.
"""
display_answer("Final answer 6", FINAL_ANSWER_6)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 7: At what dataset size or query shape would you move from local processing to a cluster?
FINAL_ANSWER_7 = """
TODO: Write your answer here. State a concrete decision boundary supported by your measurements.
"""
display_answer("Final answer 7", FINAL_ANSWER_7)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 8: How did Pandas default backend compare with the PyArrow dtype backend?
FINAL_ANSWER_8 = """
TODO: Write your answer here. Mention runtime, memory, dtypes, and whether string-heavy or IO-heavy queries changed the result.
"""
display_answer("Final answer 8", FINAL_ANSWER_8)
